# Traveling Salesperson Problem

In [24]:
"""Simple Travelling Salesperson Problem (TSP) between cities."""

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def create_data_model():
    """Stores the data for the problem."""
    data = {}
    data["distance_matrix"] = [
        [0, 2451, 713, 1018, 1631, 1374, 2408, 213, 2571, 875, 1420, 2145, 1972],
        [2451, 0, 1745, 1524, 831, 1240, 959, 2596, 403, 1589, 1374, 357, 579],
        [713, 1745, 0, 355, 920, 803, 1737, 851, 1858, 262, 940, 1453, 1260],
        [1018, 1524, 355, 0, 700, 862, 1395, 1123, 1584, 466, 1056, 1280, 987],
        [1631, 831, 920, 700, 0, 663, 1021, 1769, 949, 796, 879, 586, 371],
        [1374, 1240, 803, 862, 663, 0, 1681, 1551, 1765, 547, 225, 887, 999],
        [2408, 959, 1737, 1395, 1021, 1681, 0, 2493, 678, 1724, 1891, 1114, 701],
        [213, 2596, 851, 1123, 1769, 1551, 2493, 0, 2699, 1038, 1605, 2300, 2099],
        [2571, 403, 1858, 1584, 949, 1765, 678, 2699, 0, 1744, 1645, 653, 600],
        [875, 1589, 262, 466, 796, 547, 1724, 1038, 1744, 0, 679, 1272, 1162],
        [1420, 1374, 940, 1056, 879, 225, 1891, 1605, 1645, 679, 0, 1017, 1200],
        [2145, 357, 1453, 1280, 586, 887, 1114, 2300, 653, 1272, 1017, 0, 504],
        [1972, 579, 1260, 987, 371, 999, 701, 2099, 600, 1162, 1200, 504, 0],
    ]
    data["num_vehicles"] = 4
    data["starts"] = [0] # 起点, 此例为NY
    data["ends"] = [1] # 终端，此例为LA
    return data



In [25]:

def print_solution(manager, routing, solution):
    """Prints solution on console."""
    print(f"Objective: {solution.ObjectiveValue()} miles")
    index = routing.Start(0)
    plan_output = "Route for vehicle 0:\n"
    route_distance = 0
    while not routing.IsEnd(index):
        plan_output += f" {manager.IndexToNode(index)} ->"
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)
    plan_output += f" {manager.IndexToNode(index)}\n"
    plan_output += f"Route distance: {route_distance}miles\n"
    print(plan_output)


def main():
    """Entry point of the program."""
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager.
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]), data["num_vehicles"], data["starts"], data["ends"]
    )

    # Create Routing Model.
    routing = pywrapcp.RoutingModel(manager)


    def distance_callback(from_index, to_index):
        """Returns the distance between the two nodes."""
        # Convert from routing variable Index to distance matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data["distance_matrix"][from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    # Define cost of each arc.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Setting first solution heuristic.
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    # Solve the problem.
    solution = routing.SolveWithParameters(search_parameters)

    # Print solution on console.
    if solution:
        print_solution(manager, routing, solution)


if __name__ == "__main__":
    main()

: 

In [1]:
"""
问题描述：
    给定一批二维坐标点，每个点代表一个地点。
    只有一辆车，从 0 号点出发，访问所有点一次，最后回到 0 号点。
    目标是使总路线距离尽可能短。

核心概念：
    1. locations:
        存储所有地点的二维坐标。

    2. distance_matrix:
        存储任意两个节点之间的欧几里得距离。

    3. RoutingIndexManager:
        OR-Tools 内部索引和用户节点编号之间的转换器。

    4. RoutingModel:
        OR-Tools 路由模型，用于构建和求解路径规划问题。

    5. distance_callback:
        距离回调函数，告诉 OR-Tools 任意两个节点之间的行驶成本。

    6. ObjectiveValue:
        当前模型中，目标函数值就是整条路线的总距离。
        路线包含：
            起点 -> 中间节点 -> ... -> 最后节点 -> 起点

注意：
    - 本代码计算的是欧几里得直线距离，不是真实道路距离。
    - 距离使用 int() 向下取整，因为 OR-Tools 路由求解器通常要求成本为整数。
    - 如果你得到的 Objective 与官方示例不同，例如 2784 而不是 2790，
      通常不是错误。OR-Tools 的 Routing 求解器常使用启发式方法，
      不同版本、不同默认参数、不同搜索策略都可能得到不同结果。
      如果你的 Objective 更小，例如 2784 < 2790，说明求解器找到了更短路线。
    - 最后一个节点回到起点的距离已经在 while 循环最后一次迭代中被累加。
"""

import math

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2


def create_data_model():
    """
    创建并返回问题数据。

    返回的 data 字典包含：
        data["locations"]:
            所有地点的二维坐标列表。

        data["num_vehicles"]:
            车辆数量。这里为 1，表示单车辆问题。

        data["depot"]:
            仓库/起点/终点的节点编号。这里为 0。

    因为只有一辆车，并且车辆从 0 号点出发，最终回到 0 号点，
    所以该问题本质上是一个旅行商问题 TSP。
    """

    data = {}

    # locations 中每个元素都是一个二维坐标，例如 (288, 149)。
    #
    # 列表下标就是节点编号：
    #     locations[0] = (288, 149)
    #     locations[1] = (288, 129)
    #     locations[2] = (270, 133)
    #
    # 因此：
    #     节点 0 的坐标是 (288, 149)
    #     节点 1 的坐标是 (288, 129)
    #
    # 这些坐标单位是 block units，也就是街区单位或网格单位。
    # 不是严格意义上的米，所以后面打印 m 只是示例写法。
    data["locations"] = [
        # fmt: off
        (288, 149), (288, 129), (270, 133), (256, 141), (256, 157), (246, 157),
        (236, 169), (228, 169), (228, 161), (220, 169), (212, 169), (204, 169),
        (196, 169), (188, 169), (196, 161), (188, 145), (172, 145), (164, 145),
        (156, 145), (148, 145), (140, 145), (148, 169), (164, 169), (172, 169),
        (156, 169), (140, 169), (132, 169), (124, 169), (116, 161), (104, 153),
        (104, 161), (104, 169), (90, 165), (80, 157), (64, 157), (64, 165),
        (56, 169), (56, 161), (56, 153), (56, 145), (56, 137), (56, 129),
        (56, 121), (40, 121), (40, 129), (40, 137), (40, 145), (40, 153),
        (40, 161), (40, 169), (32, 169), (32, 161), (32, 153), (32, 145),
        (32, 137), (32, 129), (32, 121), (32, 113), (40, 113), (56, 113),
        (56, 105), (48, 99), (40, 99), (32, 97), (32, 89), (24, 89),
        (16, 97), (16, 109), (8, 109), (8, 97), (8, 89), (8, 81),
        (8, 73), (8, 65), (8, 57), (16, 57), (8, 49), (8, 41),
        (24, 45), (32, 41), (32, 49), (32, 57), (32, 65), (32, 73),
        (32, 81), (40, 83), (40, 73), (40, 63), (40, 51), (44, 43),
        (44, 35), (44, 27), (32, 25), (24, 25), (16, 25), (16, 17),
        (24, 17), (32, 17), (44, 11), (56, 9), (56, 17), (56, 25),
        (56, 33), (56, 41), (64, 41), (72, 41), (72, 49), (56, 49),
        (48, 51), (56, 57), (56, 65), (48, 63), (48, 73), (56, 73),
        (56, 81), (48, 83), (56, 89), (56, 97), (104, 97), (104, 105),
        (104, 113), (104, 121), (104, 129), (104, 137), (104, 145), (116, 145),
        (124, 145), (132, 145), (132, 137), (140, 137), (148, 137), (156, 137),
        (164, 137), (172, 125), (172, 117), (172, 109), (172, 101), (172, 93),
        (172, 85), (180, 85), (180, 77), (180, 69), (180, 61), (180, 53),
        (172, 53), (172, 61), (172, 69), (172, 77), (164, 81), (148, 85),
        (124, 85), (124, 93), (124, 109), (124, 125), (124, 117), (124, 101),
        (104, 89), (104, 81), (104, 73), (104, 65), (104, 49), (104, 41),
        (104, 33), (104, 25), (104, 17), (92, 9), (80, 9), (72, 9),
        (64, 21), (72, 25), (80, 25), (80, 25), (80, 41), (88, 49),
        (104, 57), (124, 69), (124, 77), (132, 81), (140, 65), (132, 61),
        (124, 61), (124, 53), (124, 45), (124, 37), (124, 29), (132, 21),
        (124, 21), (120, 9), (128, 9), (136, 9), (148, 9), (162, 9),
        (156, 25), (172, 21), (180, 21), (180, 29), (172, 29), (172, 37),
        (172, 45), (180, 45), (180, 37), (188, 41), (196, 49), (204, 57),
        (212, 65), (220, 73), (228, 69), (228, 77), (236, 77), (236, 69),
        (236, 61), (228, 61), (228, 53), (236, 53), (236, 45), (228, 45),
        (228, 37), (236, 37), (236, 29), (228, 29), (228, 21), (236, 21),
        (252, 21), (260, 29), (260, 37), (260, 45), (260, 53), (260, 61),
        (260, 69), (260, 77), (276, 77), (276, 69), (276, 61), (276, 53),
        (284, 53), (284, 61), (284, 69), (284, 77), (284, 85), (284, 93),
        (284, 101), (288, 109), (280, 109), (276, 101), (276, 93), (276, 85),
        (268, 97), (260, 109), (252, 101), (260, 93), (260, 85), (236, 85),
        (228, 85), (228, 93), (236, 93), (236, 101), (228, 101), (228, 109),
        (228, 117), (228, 125), (220, 125), (212, 117), (204, 109), (196, 101),
        (188, 93), (180, 93), (180, 101), (180, 109), (180, 117), (180, 125),
        (196, 145), (204, 145), (212, 145), (220, 145), (228, 145), (236, 145),
        (246, 141), (252, 125), (260, 129), (280, 133)
        # fmt: on
    ]

    # 只有 1 辆车。
    # 当车辆数量为 1 时，并且要求访问所有节点再回到起点，
    # 问题就是典型 TSP。
    data["num_vehicles"] = 4

    # depot 表示仓库，也就是起点和终点。
    # 这里设置为 0，表示车辆从节点 0 出发，最后回到节点 0。
    data["depot"] = 0

    return data


def compute_euclidean_distance_matrix(locations):
    """
    根据所有坐标点计算欧几里得距离矩阵。

    参数：
        locations:
            二维坐标列表，例如：
                [(288, 149), (288, 129), ...]

    返回：
        distances:
            一个二维字典。
            distances[i][j] 表示节点 i 到节点 j 的距离。

    示例：
        distances[0][1] 表示节点 0 到节点 1 的距离。

    欧几里得距离公式：
        distance = sqrt((x1 - x2)^2 + (y1 - y2)^2)

    在代码中使用：
        math.hypot(dx, dy)

    注意：
        OR-Tools RoutingModel 的成本通常需要整数。
        因此这里使用 int() 把距离转为整数。
        int() 是向下取整，不是四舍五入。

        例如：
            int(12.9) == 12

        如果你想四舍五入，可以改成：
            round(math.hypot(dx, dy))
    """

    distances = {}

    # 外层循环：遍历每一个出发节点。
    for from_counter, from_node in enumerate(locations):
        distances[from_counter] = {}

        # 内层循环：遍历每一个目标节点。
        for to_counter, to_node in enumerate(locations):
            # 同一个节点到自身的距离为 0。
            if from_counter == to_counter:
                distances[from_counter][to_counter] = 0
            else:
                # 计算横向差值 dx 和纵向差值 dy。
                dx = from_node[0] - to_node[0]
                dy = from_node[1] - to_node[1]

                # math.hypot(dx, dy) 等价于 sqrt(dx * dx + dy * dy)。
                # int(...) 将浮点距离转成整数。
                distances[from_counter][to_counter] = int(math.hypot(dx, dy))

    return distances


def print_solution(manager, routing, solution, show_each_arc=False):
    """
    打印 OR-Tools 求得的路线结果。

    参数：
        manager:
            RoutingIndexManager 对象。
            用于在 OR-Tools 内部索引和用户节点编号之间转换。

        routing:
            RoutingModel 对象。
            存储路由模型、成本函数和变量。

        solution:
            求解器返回的解。

        show_each_arc:
            是否打印每一条边的距离。
            如果设置为 True，会输出类似：
                0 -> 1, cost = 20

    重点解释：
        很多人会误以为最后一个节点回到起点的距离没有被累加。
        实际上，下面的 while 循环已经把这段距离加上了。

        假设路线是：
            0 -> 5 -> 8 -> 3 -> 0

        while 循环的累加过程是：
            第一次：加 0 -> 5
            第二次：加 5 -> 8
            第三次：加 8 -> 3
            第四次：加 3 -> 0

        第四次循环时：
            previous_index 是最后一个访问节点 3；
            index 被更新为终点，也就是 depot 0；
            GetArcCostForVehicle(previous_index, index, 0)
            正好计算的就是 3 -> 0 的距离。

        因此最后返回起点的距离没有遗漏。
    """

    # solution.ObjectiveValue() 是求解器计算出来的目标函数值。
    # 在当前模型中，目标函数就是整条路线的总距离。
    print(f"Solver objective: {solution.ObjectiveValue()}")

    # 获取第 0 辆车的起始内部索引。
    # 当前问题只有一辆车，所以车辆编号是 0。
    index = routing.Start(0)

    plan_output = "Route:\n"
    route_distance = 0

    # 只要当前 index 不是终点，就继续沿着解中的 NextVar 向后走。
    while not routing.IsEnd(index):
        # from_index 是 OR-Tools 内部索引。
        from_index = index

        # from_node 是用户定义的节点编号。
        # manager.IndexToNode 用于把 OR-Tools 内部索引转为用户节点编号。
        from_node = manager.IndexToNode(from_index)

        # 在路线字符串中记录当前节点。
        plan_output += f" {from_node} ->"

        # routing.NextVar(from_index) 是一个变量，
        # 表示 from_index 后面应该访问哪个索引。
        #
        # solution.Value(...) 用于读取这个变量在最终解中的取值。
        to_index = solution.Value(routing.NextVar(from_index))

        # to_node 是下一个用户节点编号。
        to_node = manager.IndexToNode(to_index)

        # 计算 from_index -> to_index 这条边对于第 0 辆车的成本。
        #
        # 因为前面调用了：
        #     routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
        #
        # 所以这里的 arc_cost 就是距离回调函数返回的距离。
        arc_cost = routing.GetArcCostForVehicle(from_index, to_index, 0)

        # 累加路线距离。
        # 注意：
        #     当 from_index 是最后一个客户节点，
        #     to_index 会是终点 depot。
        #     所以最后节点回到起点的距离也在这里累加了。
        route_distance += arc_cost

        if show_each_arc:
            print(f"{from_node} -> {to_node}, cost = {arc_cost}")

        # 移动到下一个索引，继续循环。
        index = to_index

    # while 结束时，index 已经是终点索引。
    # 对于当前问题，终点节点通常也是 depot，也就是节点 0。
    plan_output += f" {manager.IndexToNode(index)}\n"

    # 这里的 route_distance 是手动沿路线累加得到的距离。
    # 它应当与 solution.ObjectiveValue() 一致。
    #
    # 使用 block units 更严谨，因为坐标单位并不一定是米。
    plan_output += f"Manual route distance: {route_distance} block units\n"

    print(plan_output)

    # 额外打印一次对比，方便检查是否漏算。
    print(f"Manual route distance: {route_distance}")
    print(f"Solver objective:       {solution.ObjectiveValue()}")

    if route_distance == solution.ObjectiveValue():
        print("Check: manual distance equals solver objective.")
    else:
        print(
            "Warning: manual distance differs from solver objective. "
            "This may happen if extra costs, penalties, or constraints are added."
        )


def main():
    """
    主函数。

    执行流程：
        1. 创建问题数据。
        2. 计算距离矩阵。
        3. 创建 RoutingIndexManager。
        4. 创建 RoutingModel。
        5. 定义距离回调函数。
        6. 注册距离回调函数。
        7. 设置车辆行驶成本。
        8. 设置搜索参数。
        9. 求解。
        10. 打印结果。
    """

    # 1. 创建数据。
    data = create_data_model()

    # 2. 计算所有节点之间的距离矩阵。
    distance_matrix = compute_euclidean_distance_matrix(data["locations"])

    # 3. 创建 OR-Tools 索引管理器。
    #
    # OR-Tools 内部有自己的一套索引，叫 Routing variable Index。
    # 用户坐标列表中的下标叫 NodeIndex。
    #
    # 二者不一定总是相同，所以需要 manager 进行转换：
    #     manager.IndexToNode(index)
    #     manager.NodeToIndex(node)
    #
    # 参数含义：
    #     len(data["locations"])  节点数量
    #     data["num_vehicles"]   车辆数量
    #     data["depot"]          仓库/起点/终点节点编号
    manager = pywrapcp.RoutingIndexManager(
        len(data["locations"]),
        data["num_vehicles"],
        data["depot"],
    )

    # 4. 创建路由模型。
    routing = pywrapcp.RoutingModel(manager)

    # 5. 定义距离回调函数。
    #
    # OR-Tools 在求解时会不断询问：
    #     从 from_index 到 to_index 的成本是多少？
    #
    # 注意：
    #     from_index 和 to_index 是 OR-Tools 内部索引，
    #     不能直接拿来访问 distance_matrix。
    #
    # 所以必须先用 manager.IndexToNode 转为用户节点编号。
    def distance_callback(from_index, to_index):
        """
        返回两个 OR-Tools 内部索引对应节点之间的距离。
        """

        # 将 OR-Tools 内部索引转换为用户节点编号。
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        # 从预先计算好的距离矩阵中取距离。
        return distance_matrix[from_node][to_node]

    # 6. 注册距离回调函数。
    #
    # OR-Tools 不直接使用 Python 函数对象作为成本函数，
    # 而是需要先注册，注册后返回一个 callback index。
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    # 7. 设置所有车辆的边成本。
    #
    # 当前只有 1 辆车，所以这句话可以理解为：
    #     这辆车从一个节点走到另一个节点的成本
    #     由 distance_callback 计算。
    #
    # 求解器的目标就是最小化所有边成本之和。
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # 8. 设置搜索参数。
    #
    # 如果直接使用：
    #     pywrapcp.DefaultRoutingSearchParameters()
    #
    # 不同 OR-Tools 版本可能使用不同默认策略，
    # 从而得到与官方示例不同的 Objective。
    #
    # 官方 TSP 示例常使用 PATH_CHEAPEST_ARC 作为初始解策略。
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()

    # PATH_CHEAPEST_ARC:
    #     从起点开始，每一步选择当前看起来最便宜的边，
    #     构造一条初始路线。
    #
    # 使用该策略更接近官方示例设置。
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    # 如果你想让求解器继续改进解，可以启用局部搜索。
    # 启用后，结果可能比官方 2790 更小，例如 2784。
    #
    # 注意：
    #     启用局部搜索后，结果更可能与官方示例输出不同，
    #     但通常路线会更好。
    #
    # search_parameters.local_search_metaheuristic = (
    #     routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    # )
    # search_parameters.time_limit.seconds = 30

    # 如果想查看搜索日志，可以打开：
    # search_parameters.log_search = True


    # 9. 求解。
    solution = routing.SolveWithParameters(search_parameters)

    # 10. 打印结果。
    #
    # 求解器不一定总能找到解，所以要先判断 solution 是否为空。
    if solution:
        # show_each_arc=True 可以逐边打印，验证最后节点回起点的距离已累加。
        print_solution(
            manager=manager,
            routing=routing,
            solution=solution,
            show_each_arc=False,
        )
    else:
        print("No solution found.")


# Python 文件入口。
# 只有直接运行本文件时，才会执行 main()。
# 如果本文件被其他模块 import，则不会自动执行 main()。
if __name__ == "__main__":
    main()


Solver objective: 2693
Route:
 0 -> 0
Manual route distance: 0 block units

Manual route distance: 0
Solver objective:       2693


In [2]:
"""Simple Vehicles Routing Problem (VRP).

This is a sample using the routing library python wrapper to solve a VRP
problem.
A description of the problem can be found here:
http://en.wikipedia.org/wiki/Vehicle_routing_problem.

Distances are in meters.
"""

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp



def create_data_model():
    """Stores the data for the problem."""
    data = {}
    data["distance_matrix"] = [
        # fmt: off
      [0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536, 502, 388, 354, 468, 776, 662],
      [548, 0, 684, 308, 194, 502, 730, 354, 696, 742, 1084, 594, 480, 674, 1016, 868, 1210],
      [776, 684, 0, 992, 878, 502, 274, 810, 468, 742, 400, 1278, 1164, 1130, 788, 1552, 754],
      [696, 308, 992, 0, 114, 650, 878, 502, 844, 890, 1232, 514, 628, 822, 1164, 560, 1358],
      [582, 194, 878, 114, 0, 536, 764, 388, 730, 776, 1118, 400, 514, 708, 1050, 674, 1244],
      [274, 502, 502, 650, 536, 0, 228, 308, 194, 240, 582, 776, 662, 628, 514, 1050, 708],
      [502, 730, 274, 878, 764, 228, 0, 536, 194, 468, 354, 1004, 890, 856, 514, 1278, 480],
      [194, 354, 810, 502, 388, 308, 536, 0, 342, 388, 730, 468, 354, 320, 662, 742, 856],
      [308, 696, 468, 844, 730, 194, 194, 342, 0, 274, 388, 810, 696, 662, 320, 1084, 514],
      [194, 742, 742, 890, 776, 240, 468, 388, 274, 0, 342, 536, 422, 388, 274, 810, 468],
      [536, 1084, 400, 1232, 1118, 582, 354, 730, 388, 342, 0, 878, 764, 730, 388, 1152, 354],
      [502, 594, 1278, 514, 400, 776, 1004, 468, 810, 536, 878, 0, 114, 308, 650, 274, 844],
      [388, 480, 1164, 628, 514, 662, 890, 354, 696, 422, 764, 114, 0, 194, 536, 388, 730],
      [354, 674, 1130, 822, 708, 628, 856, 320, 662, 388, 730, 308, 194, 0, 342, 422, 536],
      [468, 1016, 788, 1164, 1050, 514, 514, 662, 320, 274, 388, 650, 536, 342, 0, 764, 194],
      [776, 868, 1552, 560, 674, 1050, 1278, 742, 1084, 810, 1152, 274, 388, 422, 764, 0, 798],
      [662, 1210, 754, 1358, 1244, 708, 480, 856, 514, 468, 354, 844, 730, 536, 194, 798, 0],
        # fmt: on
    ]
    data["num_vehicles"] = 4
    data["depot"] = 0
    return data


def print_solution(data, manager, routing, solution):
    """Prints solution on console."""
    print(f"Objective: {solution.ObjectiveValue()}")
    max_route_distance = 0
    for vehicle_id in range(data["num_vehicles"]):
        if not routing.IsVehicleUsed(solution, vehicle_id):
            continue
        index = routing.Start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        route_distance = 0
        while not routing.IsEnd(index):
            plan_output += f" {manager.IndexToNode(index)} -> "
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )
        plan_output += f"{manager.IndexToNode(index)}\n"
        plan_output += f"Distance of the route: {route_distance}m\n"
        print(plan_output)
        max_route_distance = max(route_distance, max_route_distance)
    print(f"Maximum of the route distances: {max_route_distance}m")



def main():
    """Entry point of the program."""
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager.
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]), data["num_vehicles"], data["depot"]
    )

    # Create Routing Model.
    routing = pywrapcp.RoutingModel(manager)

    # Create and register a transit callback.
    def distance_callback(from_index, to_index):
        """Returns the distance between the two nodes."""
        # Convert from routing variable Index to distance matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data["distance_matrix"][from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    # Define cost of each arc.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Add Distance constraint.
    dimension_name = "Distance"
    routing.AddDimension(
        transit_callback_index,
        0,  # no slack
        3000,  # vehicle maximum travel distance
        True,  # start cumul to zero
        dimension_name,
    )
    distance_dimension = routing.GetDimensionOrDie(dimension_name)
    distance_dimension.SetGlobalSpanCostCoefficient(100)

    # Setting first solution heuristic.
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    # Solve the problem.
    solution = routing.SolveWithParameters(search_parameters)

    # Print solution on console.
    if solution:
        print_solution(data, manager, routing, solution)
    else:
        print("No solution found !")


if __name__ == "__main__":
    main()

Objective: 177500
Route for vehicle 0:
 0 ->  9 ->  10 ->  2 ->  6 ->  5 -> 0
Distance of the route: 1712m

Route for vehicle 1:
 0 ->  16 ->  14 ->  8 -> 0
Distance of the route: 1484m

Route for vehicle 2:
 0 ->  7 ->  1 ->  4 ->  3 -> 0
Distance of the route: 1552m

Route for vehicle 3:
 0 ->  13 ->  15 ->  11 ->  12 -> 0
Distance of the route: 1552m

Maximum of the route distances: 1712m
